<a href="https://colab.research.google.com/github/cheonyoungho/Math_Box/blob/master/Trnasformer_de_en.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install datasets


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 484.9/484.9 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 11.8 MB/s eta 0:00:00


In [25]:
import math
import re
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset

In [26]:
# 1. 유틸리티 함수: 마스크 생성 함수 (utils 대체)
# =======================================================
def create_pad_mask(seq, pad_idx):
    # seq: [batch_size, seq_len]
    return (seq == pad_idx).unsqueeze(1)  # [batch_size, 1, seq_len]

def create_trg_self_mask(size, device):
    # causal mask: (size x size)에서 위 삼각 부분을 True로
    mask = torch.triu(torch.ones((size, size), device=device), diagonal=1).bool()
    # [size, size] --> [1, 1, size, size]
    # 이렇게 만들면 (batch, head, seq_len, seq_len)과 쉽게 broadcast 가능
    mask = mask.unsqueeze(0).unsqueeze(0)
    return mask

# =======================================================
# 2. 간단한 토크나이저 및 Vocabulary 클래스
# =======================================================
class Vocabulary:
    def __init__(self, freq_threshold=2):
        self.freq_threshold = freq_threshold
        # 특수 토큰 정의
        self.pad_token = "<pad>"
        self.sos_token = "<sos>"
        self.eos_token = "<eos>"
        self.unk_token = "<unk>"

        self.token2idx = {
            self.pad_token: 0,
            self.sos_token: 1,
            self.eos_token: 2,
            self.unk_token: 3
        }
        self.idx2token = {idx: token for token, idx in self.token2idx.items()}

    def tokenizer(self, text):
        # 소문자화, 공백 기준 토큰화, 구두점 분리 등
        text = text.lower().strip()
        text = re.sub(r"([?.!,])", r" \1 ", text)
        text = re.sub(r'[" "]+', " ", text)
        text = re.sub(r"[^a-zA-Z?.!,]+", " ", text)
        return text.split()

    def build_vocabulary(self, sentence_list):
        frequencies = Counter()
        for sentence in sentence_list:
            tokens = self.tokenizer(sentence)
            frequencies.update(tokens)
        for word, freq in frequencies.items():
            if freq >= self.freq_threshold and word not in self.token2idx:
                idx = len(self.token2idx)
                self.token2idx[word] = idx
                self.idx2token[idx] = word

    def numericalize(self, text):
        tokenized_text = self.tokenizer(text)
        return [self.token2idx.get(token, self.token2idx[self.unk_token]) for token in tokenized_text]


In [27]:
# =======================================================
# 3. Dataset 클래스 및 collate 함수
# =======================================================

# 입력,출력문장을 단어사전에 넣어서 인덱스를 부여
class TranslationDataset(Dataset):
    def __init__(self, hf_dataset, src_lang="de", trg_lang="en",
                 src_vocab=None, trg_vocab=None, max_len=50):
        """
        hf_dataset: Hugging Face 데이터셋 (예: dataset['train'])
        src_lang, trg_lang: 데이터셋 내 translation 필드의 key : 독일어 , 영어 문장
        src_vocab, trg_vocab: Vocabulary 객체 (외부에서 생성 후 전달)
        max_len: 최대 토큰 길이
        """
        self.dataset = hf_dataset
        self.src_lang = src_lang
        self.trg_lang = trg_lang
        self.src_vocab = src_vocab
        self.trg_vocab = trg_vocab
        self.max_len = max_len

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        sample = self.dataset[idx]['translation']
        src_text = sample[self.src_lang]
        trg_text = sample[self.trg_lang]

        # 숫자 인덱스 변환 (시작/종료 토큰 추가)
        src_ids = [self.src_vocab.token2idx[self.src_vocab.sos_token]] + \
                  self.src_vocab.numericalize(src_text) + \
                  [self.src_vocab.token2idx[self.src_vocab.eos_token]]

        trg_ids = [self.trg_vocab.token2idx[self.trg_vocab.sos_token]] + \
                  self.trg_vocab.numericalize(trg_text) + \
                  [self.trg_vocab.token2idx[self.trg_vocab.eos_token]]

        # 최대 길이 제한
        src_ids = src_ids[:self.max_len]
        trg_ids = trg_ids[:self.max_len]
        return torch.tensor(src_ids, dtype=torch.long), torch.tensor(trg_ids, dtype=torch.long)


# 각각 (src_ids , trg_ids)을 하나의 미니배치로 묶을 떄 , 각 샘플의 길이가 다르면 패딩을 추가
# 모두 같은 길이로 맞추는 역할

def collate_fn(batch, pad_idx=0):
    # batch: list of (src_ids, trg_ids)
    src_batch, trg_batch = zip(*batch)
    src_lens = [len(x) for x in src_batch]
    trg_lens = [len(x) for x in trg_batch]
    max_src = max(src_lens)
    max_trg = max(trg_lens)

    padded_src = [F.pad(x, (0, max_src - len(x)), value=pad_idx) for x in src_batch]
    padded_trg = [F.pad(x, (0, max_trg - len(x)), value=pad_idx) for x in trg_batch]
    return torch.stack(padded_src), torch.stack(padded_trg)

In [30]:
# =======================================================
# 4. Transformer 모델 구현 (제공된 코드 기반)
# =======================================================
def initialize_weight(x):
    nn.init.xavier_uniform_(x.weight)
    if x.bias is not None:
        nn.init.constant_(x.bias, 0)

class FeedForwardNetwork(nn.Module):
    def __init__(self, hidden_size, filter_size, dropout_rate):
        super(FeedForwardNetwork, self).__init__()
        self.layer1 = nn.Linear(hidden_size, filter_size)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout_rate)
        self.layer2 = nn.Linear(filter_size, hidden_size)
        initialize_weight(self.layer1)
        initialize_weight(self.layer2)
    def forward(self, x):
        x = self.layer1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.layer2(x)
        return x

class MultiHeadAttention(nn.Module):
    def __init__(self, hidden_size, dropout_rate, head_count=8):
        super(MultiHeadAttention, self).__init__()
        self.head_count = head_count
        self.att_size = att_size = hidden_size // head_count  # d_k = 512/8 = 64
        self.scale = att_size ** -0.5
        self.linear_q = nn.Linear(hidden_size, head_count * att_size, bias=False)
        self.linear_k = nn.Linear(hidden_size, head_count * att_size, bias=False)
        self.linear_v = nn.Linear(hidden_size, head_count * att_size, bias=False)
        initialize_weight(self.linear_q)
        initialize_weight(self.linear_k)
        initialize_weight(self.linear_v)
        self.att_dropout = nn.Dropout(dropout_rate)
        self.output_layer = nn.Linear(head_count * att_size, hidden_size, bias=False)
        initialize_weight(self.output_layer)


    def forward(self, q, k, v, mask, cache=None):
        orig_q_size = q.size()
        d_k = self.att_size
        d_v = self.att_size
        batch_size = q.size(0)

        q = self.linear_q(q).view(batch_size, -1, self.head_count, d_k)
        if cache is not None and 'encdec_k' in cache:
            k, v = cache['encdec_k'], cache['encdec_v']
        else:
            k = self.linear_k(k).view(batch_size, -1, self.head_count, d_k)
            v = self.linear_v(v).view(batch_size, -1, self.head_count, d_v)
            if cache is not None:
                cache['encdec_k'], cache['encdec_v'] = k, v

        q = q.transpose(1, 2)                  # [b, h, q_len, d_k]
        v = v.transpose(1, 2)                  # [b, h, v_len, d_v]
        k = k.transpose(1, 2).transpose(2, 3)  # [b, h, d_k, k_len]

        q.mul_(self.scale)
        x = torch.matmul(q, k)  # [b, h, q_len, k_len]
        x.masked_fill_(mask, float('-inf'))
        x = torch.softmax(x, dim=3)
        x = self.att_dropout(x)
        x = x.matmul(v)  # [b, h, q_len, d_v]
        x = x.transpose(1, 2).contiguous()  # [b, q_len, h, d_v]
        x = x.view(batch_size, -1, self.head_count * d_v)
        x = self.output_layer(x)
        assert x.size() == orig_q_size
        return x

        q = self.linear_q(q).view(batch_size, -1, self.head_count, d_k)
        if cache is not None and 'encdec_k' in cache:
            k, v = cache['encdec_k'], cache['encdec_v']
        else:
            k = self.linear_k(k).view(batch_size, -1, self.head_count, d_k)
            v = self.linear_v(v).view(batch_size, -1, self.head_count, d_v)
            if cache is not None:
                cache['encdec_k'], cache['encdec_v'] = k, v
        q = q.transpose(1, 2)                  # [b, h, q_len, d_k]
        v = v.transpose(1, 2)                  # [b, h, v_len, d_v]
        k = k.transpose(1, 2).transpose(2, 3)  # [b, h, d_k, k_len]
        q.mul_(self.scale)
        x = torch.matmul(q, k)  # [b, h, q_len, k_len]
        x.masked_fill_(mask.unsqueeze(1), -1e9)
        x = torch.softmax(x, dim=3)
        x = self.att_dropout(x)
        x = x.matmul(v)  # [b, h, q_len, d_v]
        x = x.transpose(1, 2).contiguous()  # [b, q_len, h, d_v]
        x = x.view(batch_size, -1, self.head_count * d_v)
        x = self.output_layer(x)
        assert x.size() == orig_q_size
        return x


class EncoderLayer(nn.Module):
    def __init__(self, hidden_size, filter_size, dropout_rate):
        super(EncoderLayer, self).__init__()
        self.self_attention_norm = nn.LayerNorm(hidden_size, eps=1e-6)
        self.self_attention = MultiHeadAttention(hidden_size, dropout_rate, head_count=8)
        self.self_attention_dropout = nn.Dropout(dropout_rate)
        self.ffn_norm = nn.LayerNorm(hidden_size, eps=1e-6)
        self.ffn = FeedForwardNetwork(hidden_size, filter_size, dropout_rate)
        self.ffn_dropout = nn.Dropout(dropout_rate)
    def forward(self, x, mask):
        y = self.self_attention_norm(x)
        y = self.self_attention(y, y, y, mask)
        y = self.self_attention_dropout(y)
        x = x + y
        y = self.ffn_norm(x)
        y = self.ffn(y)
        y = self.ffn_dropout(y)
        x = x + y
        return x

class DecoderLayer(nn.Module):
    def __init__(self, hidden_size, filter_size, dropout_rate):
        super(DecoderLayer, self).__init__()
        self.self_attention_norm = nn.LayerNorm(hidden_size, eps=1e-6)
        self.self_attention = MultiHeadAttention(hidden_size, dropout_rate, head_count=8)
        self.self_attention_dropout = nn.Dropout(dropout_rate)
        self.enc_dec_attention_norm = nn.LayerNorm(hidden_size, eps=1e-6)
        self.enc_dec_attention = MultiHeadAttention(hidden_size, dropout_rate, head_count=8)
        self.enc_dec_attention_dropout = nn.Dropout(dropout_rate)
        self.ffn_norm = nn.LayerNorm(hidden_size, eps=1e-6)
        self.ffn = FeedForwardNetwork(hidden_size, filter_size, dropout_rate)
        self.ffn_dropout = nn.Dropout(dropout_rate)
    def forward(self, x, enc_output, self_mask, i_mask, cache):
        y = self.self_attention_norm(x)
        y = self.self_attention(y, y, y, self_mask)
        y = self.self_attention_dropout(y)
        x = x + y
        if enc_output is not None:
            y = self.enc_dec_attention_norm(x)
            y = self.enc_dec_attention(y, enc_output, enc_output, i_mask, cache)
            y = self.enc_dec_attention_dropout(y)
            x = x + y
        y = self.ffn_norm(x)
        y = self.ffn(y)
        y = self.ffn_dropout(y)
        x = x + y
        return x


class Encoder(nn.Module):
    def __init__(self, hidden_size, filter_size, dropout_rate, n_layers):
        super(Encoder, self).__init__()
        encoders = [EncoderLayer(hidden_size, filter_size, dropout_rate)
                    for _ in range(n_layers)]
        self.layers = nn.ModuleList(encoders)
        self.last_norm = nn.LayerNorm(hidden_size, eps=1e-6)
    def forward(self, inputs, mask):
        encoder_output = inputs
        for enc_layer in self.layers:
            encoder_output = enc_layer(encoder_output, mask)
        return self.last_norm(encoder_output)

class Decoder(nn.Module):
    def __init__(self, hidden_size, filter_size, dropout_rate, n_layers):
        super(Decoder, self).__init__()
        decoders = [DecoderLayer(hidden_size, filter_size, dropout_rate)
                    for _ in range(n_layers)]
        self.layers = nn.ModuleList(decoders)
        self.last_norm = nn.LayerNorm(hidden_size, eps=1e-6)
    def forward(self, targets, enc_output, i_mask, t_self_mask, cache):
        decoder_output = targets
        for i, dec_layer in enumerate(self.layers):
            layer_cache = None
            if cache is not None:
                if i not in cache:
                    cache[i] = {}
                layer_cache = cache[i]
            decoder_output = dec_layer(decoder_output, enc_output, t_self_mask, i_mask, layer_cache)
        return self.last_norm(decoder_output)

class Transformer(nn.Module):
    def __init__(self, i_vocab_size, t_vocab_size,
                 n_layers=6,
                 hidden_size=512,
                 filter_size=2048,
                 dropout_rate=0.1,
                 share_target_embedding=True,
                 has_inputs=True,
                 src_pad_idx=None,
                 trg_pad_idx=None):
        super(Transformer, self).__init__()
        self.hidden_size = hidden_size
        self.emb_scale = hidden_size ** 0.5
        self.has_inputs = has_inputs
        self.src_pad_idx = src_pad_idx
        self.trg_pad_idx = trg_pad_idx

        self.t_vocab_embedding = nn.Embedding(t_vocab_size, hidden_size)
        nn.init.normal_(self.t_vocab_embedding.weight, mean=0, std=hidden_size**-0.5)
        self.t_emb_dropout = nn.Dropout(dropout_rate)
        self.decoder = Decoder(hidden_size, filter_size, dropout_rate, n_layers)

        if has_inputs:
            if not share_target_embedding:
                self.i_vocab_embedding = nn.Embedding(i_vocab_size, hidden_size)
                nn.init.normal_(self.i_vocab_embedding.weight, mean=0, std=hidden_size**-0.5)
            else:
                self.i_vocab_embedding = self.t_vocab_embedding
            self.i_emb_dropout = nn.Dropout(dropout_rate)
            self.encoder = Encoder(hidden_size, filter_size, dropout_rate, n_layers)

        # Positional encoding
        num_timescales = self.hidden_size // 2
        max_timescale = 10000.0
        min_timescale = 1.0
        log_timescale_increment = math.log(max_timescale/min_timescale) / max(num_timescales - 1, 1)
        inv_timescales = min_timescale * torch.exp(torch.arange(num_timescales, dtype=torch.float32) * -log_timescale_increment)
        self.register_buffer('inv_timescales', inv_timescales)

    def forward(self, inputs, targets):
        enc_output, i_mask = None, None
        if self.has_inputs:
            i_mask = create_pad_mask(inputs, self.src_pad_idx)
            enc_output = self.encode(inputs, i_mask)
        t_mask = create_pad_mask(targets, self.trg_pad_idx)
        target_size = targets.size(1)
        t_self_mask = create_trg_self_mask(target_size, device=targets.device)
        return self.decode(targets, enc_output, i_mask, t_self_mask, t_mask)

    def encode(self, inputs, i_mask):
        input_embedded = self.i_vocab_embedding(inputs)
        input_embedded.masked_fill_(i_mask.squeeze(1).unsqueeze(-1), 0)
        input_embedded *= self.emb_scale
        input_embedded += self.get_position_encoding(inputs)
        input_embedded = self.i_emb_dropout(input_embedded)
        return self.encoder(input_embedded, i_mask)

    def decode(self, targets, enc_output, i_mask, t_self_mask, t_mask, cache=None):
        target_embedded = self.t_vocab_embedding(targets)
        target_embedded.masked_fill_(t_mask.squeeze(1).unsqueeze(-1), 0)
        # Shifting: 오른쪽으로 시프트 (시작 토큰을 유지)
        target_embedded = target_embedded[:, :-1]
        target_embedded = F.pad(target_embedded, (0, 0, 1, 0))
        target_embedded *= self.emb_scale
        target_embedded += self.get_position_encoding(targets)
        target_embedded = self.t_emb_dropout(target_embedded)
        decoder_output = self.decoder(target_embedded, enc_output, i_mask, t_self_mask, cache)
        output = torch.matmul(decoder_output, self.t_vocab_embedding.weight.transpose(0, 1))
        return output

    def get_position_encoding(self, x):
        max_length = x.size(1)
        position = torch.arange(max_length, dtype=torch.float32, device=x.device)
        scaled_time = position.unsqueeze(1) * self.inv_timescales.unsqueeze(0)
        signal = torch.cat([torch.sin(scaled_time), torch.cos(scaled_time)], dim=1)
        signal = F.pad(signal, (0, 0, 0, self.hidden_size % 2))
        signal = signal.view(1, max_length, self.hidden_size)
        return signal

In [31]:
# =======================================================
# 5. 전체 파이프라인 실행 (데이터 로드, 학습, 번역)
# =======================================================

# Colab GPU 확인
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("사용중인 디바이스:", device)

# 5-1. 데이터셋 로드 (OPUS Books 독일어-영어)
dataset = load_dataset("opus_books", "de-en")

사용중인 디바이스: cuda


In [32]:
#==========================================
# vocab 구성
train_data = dataset['train']

# train_data 구조 살펴보기
train_data.features
train_data.column_names
i = 4  # 예: 0번째 1~3 은 <sos> ~ <untoken>
print(train_data[i]["translation"]["de"])  # 독일어 문장
print(train_data[i]["translation"]["en"])  # 영어 문장

Es war ganz unmöglich, an diesem Tage einen Spaziergang zu machen.
There was no possibility of taking a walk that day.


In [33]:
# 소스/타겟 문장 리스트 추출
# 이중 딕셔너리라 첫번째 dic 를 item 으로 해주고
# item 에서 두번째 dic인 translation 에 들어가서 'de' , 'en' 을 뽑아준다.

src_sentences = [item['translation']['de']for item in train_data]
trg_sentences = [item['translation']['en']for item in train_data]

In [34]:
# Vocabulary 객체 생성
src_vocab = Vocabulary(freq_threshold =2)
trg_vocab = Vocabulary(freq_threshold=2)

# build_vocabulary 호출
src_vocab.build_vocabulary(src_sentences)
trg_vocab.build_vocabulary(trg_sentences)

print("소스 단어 수:", len(src_vocab.token2idx))
print("타겟 단어 수:", len(trg_vocab.token2idx))

소스 단어 수: 26757
타겟 단어 수: 18378


In [35]:
# TranslationDataset 생성
train_dataset = TranslationDataset( hf_dataset = train_data ,
                                    src_lang   = "de"       ,
                                    trg_lang   = "en"       ,
                                    src_vocab  = src_vocab  ,
                                    trg_vocab  = trg_vocab  ,
                                    max_len    = 50
                                    )

# __getitem__ 으로 데이터 확인
first = train_dataset.__getitem__(idx=1)
print(first)


# DataLoader 생성

BATCH_SIZE = 32

train_loader = DataLoader(
                           train_dataset,
                           batch_size    = BATCH_SIZE ,
                           shuffle       = True       ,
                           collate_fn    = collate_fn  # 각 배치의 문장 길이를 동일하게 맞춤
                           )

(tensor([ 1, 14, 15,  2]), tensor([1, 7, 8, 2]))


In [36]:
# pad 토큰 인덱스
src_pad_idx = src_vocab.token2idx[src_vocab.pad_token]
trg_pad_idx = trg_vocab.token2idx[trg_vocab.pad_token]

# Transformer 모델 초기화 (hidden_size=512, head_count=8)
n_layers = 6
hidden_size = 512
filter_size = 2048
dropout_rate = 0.1

model = Transformer(
                    i_vocab_size  = len(src_vocab.token2idx) ,
                    t_vocab_size  = len(trg_vocab.token2idx) ,
                    n_layers      = n_layers                 ,
                    hidden_size   = hidden_size              ,
                    filter_size   = filter_size              ,
                    dropout_rate  = dropout_rate             ,
                    share_target_embedding = False           ,
                    has_inputs    = True                     ,
                    src_pad_idx   = src_pad_idx              ,
                    trg_pad_idx   = trg_pad_idx
                    ).to(device)

print("모델 구조:")
print(model)

# Optimizer와 Loss 함수 설정
import torch.optim as optim

optimizer = optim.Adam( model.parameters()  ,
                        lr    = 1e-4        ,
                        betas = (0.9, 0.98) ,
                        eps   = 1e-9
                        )
criterion = nn.CrossEntropyLoss(ignore_index=trg_pad_idx)

모델 구조:
Transformer(
  (t_vocab_embedding): Embedding(18378, 512)
  (t_emb_dropout): Dropout(p=0.1, inplace=False)
  (decoder): Decoder(
    (layers): ModuleList(
      (0-5): 6 x DecoderLayer(
        (self_attention_norm): LayerNorm((512,), eps=1e-06, elementwise_affine=True)
        (self_attention): MultiHeadAttention(
          (linear_q): Linear(in_features=512, out_features=512, bias=False)
          (linear_k): Linear(in_features=512, out_features=512, bias=False)
          (linear_v): Linear(in_features=512, out_features=512, bias=False)
          (att_dropout): Dropout(p=0.1, inplace=False)
          (output_layer): Linear(in_features=512, out_features=512, bias=False)
        )
        (self_attention_dropout): Dropout(p=0.1, inplace=False)
        (enc_dec_attention_norm): LayerNorm((512,), eps=1e-06, elementwise_affine=True)
        (enc_dec_attention): MultiHeadAttention(
          (linear_q): Linear(in_features=512, out_features=512, bias=False)
          (linear_k): Line

In [38]:
# 5-6. 학습루프 작성

NUM_EPOCHS = 2  # 예시: 2 에폭 동안 학습

for epoch in range(NUM_EPOCHS):
    model.train()
    total_loss = 0.0

    for batch_idx, (src, trg) in enumerate(train_loader): #  train_loader = DataLoader의 객체. 미니배치 단위로 데이터를 제공
        src = src.to(device)  # 소스 배치 텐서, shape: [batch_size, src_len]
        trg = trg.to(device)  # 타깃 배치 텐서, shape: [batch_size, trg_len]

        optimizer.zero_grad() # 기울기 초기화

        # Forward: 모델에 소스와 타깃을 넣어 예측값 생성
        # 출력의 shape는 [batch_size, seq_len, t_vocab_size]
        output = model(src, trg)

        # 디코더는 <sos> 토큰을 입력받아 첫 단어를 예측하므로,
        # 정답(타깃) 시퀀스와 예측 시퀀스를 한 칸씩 시프트하여 맞춰줌.
        output_for_loss = output[:, 1:].reshape(-1, output.size(-1))
        trg_for_loss = trg[:, 1:].reshape(-1)

        loss = criterion(output_for_loss, trg_for_loss)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        # 예시: 100번째 배치마다 평균 손실을 출력
        if (batch_idx + 1) % 100 == 0:
            avg_loss = total_loss / 100
            print(f"Epoch [{epoch+1}/{NUM_EPOCHS}], Step [{batch_idx+1}], Loss: {avg_loss:.4f}")
            total_loss = 0.0

print("학습 완료.")

RuntimeError: The expanded size of the tensor (8) must match the existing size (32) at non-singleton dimension 1.  Target sizes: [32, 8, 50, 50].  Tensor sizes: [32, 1, 50]